In [1]:
%pip install pandas matplotlib seaborn numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import os
import json
import numpy as np

In [3]:
def parse_result_as_df_recommended(result_file):
    with open(result_file, 'r') as f:
        lines = f.readlines()
    json_data = json.loads(('').join(lines))
    
    # Handle evaluationResult array by extracting comprehensive statistics
    if 'evaluationResult' in json_data and isinstance(json_data['evaluationResult'], list):
        eval_results = json_data['evaluationResult']
        
        # Basic statistics
        json_data['evaluationResult_count'] = len(eval_results)

        # Exact matching statistics
        exact_matches = [item for item in eval_results if item.get('doesMatchExactly', False)]
        json_data['evaluationResult_exact_matches'] = len(exact_matches) > 0
        json_data['evaluationResult_exact_count'] = len(exact_matches)
        
        # Syntactic matching statistics
        syntactic_matches = [item for item in eval_results if item.get('doesMatchSyntactically', False)]
        json_data['evaluationResult_syntactic_matches'] = len(syntactic_matches) > 0
        json_data['evaluationResult_syntactic_match_count'] = len(syntactic_matches)
        # json_data['evaluationResult_syntactic_rate'] = len(syntactic_matches) / len(eval_results) if eval_results else 0
        
        # Semantic matching statistics
        semantic_matches = [item for item in eval_results if item.get('doesMatchSemantically', False)]
        json_data['evaluationResult_semantic_matches'] = len(semantic_matches) > 0
        json_data['evaluationResult_semantic_match_count'] = len(semantic_matches)
        # json_data['evaluationResult_semantic_rate'] = len(semantic_matches) / len(eval_results) if eval_results else 0

        starts_with_suggestion_count = [item for item in eval_results if item.get('startsWithSuggestion', False)]
        json_data['evaluationResult_starts_with_suggestion_count'] = len(starts_with_suggestion_count)

        compiled_suggestions = [item for item in eval_results if item.get('doesCompile', False)]
        json_data['evaluationResult_suggestions_compiled'] = len(compiled_suggestions) > 0
        json_data['evaluationResult_suggestions_compiled_count'] = len(compiled_suggestions)
        
        # # Top suggestions
        # json_data['evaluationResult_top_suggestion'] = eval_results[0]['suggestion'] if eval_results else None
        # json_data['evaluationResult_top_rank'] = eval_results[0]['rank'] if eval_results else None
        
        # # Average rank of matches
        # if syntactic_matches:
        #     json_data['evaluationResult_avg_syntactic_rank'] = np.mean([item['rank'] for item in syntactic_matches])
        # else:
        #     json_data['evaluationResult_avg_syntactic_rank'] = None
            
        # Remove the original array to avoid DataFrame creation issues
        del json_data['evaluationResult']

        # Delete any key that starts with 'suggestionIn' to avoid redundancy
        keys_to_delete = [key for key in json_data.keys() if key.startswith('suggestionIn')]
        for key in keys_to_delete:
            del json_data[key]
    
    # Handle any other arrays in similar fashion
    for key, value in list(json_data.items()):
        if isinstance(value, list) and key != 'evaluationResult':  # Handle other potential arrays
            json_data[f'{key}_array_length'] = len(value)
            json_data[f'{key}_as_string'] = json.dumps(value)  # Keep as JSON string if needed
            del json_data[key]  # Remove original array
    
    df = pd.DataFrame(json_data, index=[0])
    return df

In [4]:
# Updated data loading code with proper JSON array handling
LLM_TEST_RESULT_DIR = '../test-results/llm/multi_term/'

llm_results_df = pd.DataFrame()    

print("Loading LLM results...")
for root, subdirs, files in os.walk(LLM_TEST_RESULT_DIR):
    for file in files:
        if file.endswith('.result.json'):
            try:
                df = parse_result_as_df_recommended(os.path.join(root, file))
                llm_results_df = pd.concat([llm_results_df, df], axis=0, ignore_index=True)
            except Exception as e:
                print(f"Error processing {file}: {e}")

print(f"LLM results loaded: {len(llm_results_df)} records")

Loading LLM results...
LLM results loaded: 106 records


In [5]:
llm_results_df.columns

Index(['modelName', 'filename', 'offset', 'term', 'line', 'character',
       'incompletionLine', 'completionList', 'completionGenerated',
       'suggestionExists', 'expectedCompletionWord', 'expectedCompletionLine',
       'elapsedTimeInMs', 'evaluationResult_count',
       'evaluationResult_exact_matches', 'evaluationResult_exact_count',
       'evaluationResult_syntactic_matches',
       'evaluationResult_syntactic_match_count',
       'evaluationResult_semantic_matches',
       'evaluationResult_semantic_match_count',
       'evaluationResult_starts_with_suggestion_count',
       'evaluationResult_suggestions_compiled',
       'evaluationResult_suggestions_compiled_count', 'expectedTerm'],
      dtype='object')

In [6]:
llm_results_df['suggestion_state'] = llm_results_df.apply(
    lambda row: 'no-suggestion' if not row['completionGenerated'] else row['suggestionExists'], axis=1)

In [7]:
llm_results_df[llm_results_df['modelName'] == 'modelo-alloy']

,modelName,filename,offset,term,line,character,incompletionLine,completionList,completionGenerated,suggestionExists,...,evaluationResult_exact_count,evaluationResult_syntactic_matches,evaluationResult_syntactic_match_count,evaluationResult_semantic_matches,evaluationResult_semantic_match_count,evaluationResult_starts_with_suggestion_count,evaluationResult_suggestions_compiled,evaluationResult_suggestions_compiled_count,expectedTerm,suggestion_state


In [8]:
import matplotlib.pyplot as plt

In [9]:
# Configure notebook display for wider figures
from IPython.display import HTML, display

# Set CSS to make figures use more width
display(HTML("""
<style>
    .output_png {
        display: block;
        margin: 0 auto;
        max-width: 100% !important;
        width: 100% !important;
    }
    .jp-OutputArea-output {
        overflow-x: auto;
    }
</style>
"""))

# Configure matplotlib backend for better notebook integration
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 0

In [10]:
# LLM ANALYSIS
llm_avg_time_per_model = llm_results_df.groupby('modelName')['elapsedTimeInMs'].mean()
llm_median_time_per_model = llm_results_df.groupby('modelName')['elapsedTimeInMs'].median()
llm_p99_time_per_model = llm_results_df.groupby('modelName')['elapsedTimeInMs'].quantile(0.99)
llm_suggestion_count_per_model = llm_results_df.groupby('modelName')['evaluationResult_count'].mean()
llm_suggestion_state_count_per_model = llm_results_df.groupby('modelName')['suggestion_state'].value_counts().unstack().fillna(0)
llm_suggestion_state_count = llm_suggestion_state_count_per_model.sum()

print("\nOverall LLM suggestion state counts:")
print(llm_suggestion_state_count)


Overall LLM suggestion state counts:
suggestion_state
False    23
True     83
dtype: int64


In [11]:
report_df = llm_results_df.groupby('modelName').size().reset_index(name='# loc')

In [12]:
report_df

,modelName,# loc
0,array,8
1,classroom-fol,38
2,courses-v2,40
3,git-redacted,20


In [13]:
total_suggestions_generated = llm_results_df[['modelName', 'evaluationResult_count']].groupby('modelName').sum().reset_index()
total_suggestions_generated['avg_suggestions_per_loc'] = total_suggestions_generated['evaluationResult_count'] / report_df['# loc']

total_suggestions_generated.rename(columns={'evaluationResult_count': 'total_suggestions_generated'}, inplace=True)
total_suggestions_generated

,modelName,total_suggestions_generated,avg_suggestions_per_loc
0,array,64,8.000000
1,classroom-fol,485,12.763158
2,courses-v2,454,11.350000
3,git-redacted,117,5.850000


In [14]:
report_df = report_df.merge(total_suggestions_generated, on='modelName', how='left')
report_df

,modelName,# loc,total_suggestions_generated,avg_suggestions_per_loc
0,array,8,64,8.000000
1,classroom-fol,38,485,12.763158
2,courses-v2,40,454,11.350000
3,git-redacted,20,117,5.850000


In [15]:
report_df = report_df.merge(llm_avg_time_per_model.reset_index().rename(columns={'elapsedTimeInMs': 'avg_time_in_ms'}), on='modelName', how='left')
report_df

,modelName,# loc,total_suggestions_generated,avg_suggestions_per_loc,avg_time_in_ms
0,array,8,64,8.000000,19142.618969
1,classroom-fol,38,485,12.763158,26219.957100
2,courses-v2,40,454,11.350000,21570.513820
3,git-redacted,20,117,5.850000,33411.818150


In [16]:
syntactic_matches_per_model = llm_results_df.groupby('modelName')['evaluationResult_syntactic_match_count'].sum()

report_df = report_df.merge(syntactic_matches_per_model.reset_index().rename(columns={'evaluationResult_syntactic_match_count': 'syntactic_matches'}), on='modelName', how='left')
report_df

,modelName,# loc,total_suggestions_generated,avg_suggestions_per_loc,avg_time_in_ms,syntactic_matches
0,array,8,64,8.000000,19142.618969,4
1,classroom-fol,38,485,12.763158,26219.957100,19
2,courses-v2,40,454,11.350000,21570.513820,18
3,git-redacted,20,117,5.850000,33411.818150,10


In [17]:
semantic_matches_per_model = llm_results_df.groupby('modelName')['evaluationResult_semantic_match_count'].sum()
semantic_matches_rate_per_model = llm_results_df.groupby('modelName').apply(lambda x: x['evaluationResult_semantic_match_count'].sum() * 100 / x['evaluationResult_count'].sum() if x['evaluationResult_count'].sum() > 0 else 0)

report_df = report_df.merge(semantic_matches_per_model.reset_index().rename(columns={'evaluationResult_semantic_match_count': 'semantic_matches'}), on='modelName', how='left')

report_df = report_df.merge(semantic_matches_rate_per_model.reset_index().rename(columns={0: 'semantic_match_rate'}), on='modelName', how='left')

starts_with_suggestion_count_per_model = llm_results_df.groupby('modelName').apply(lambda x: x['evaluationResult_starts_with_suggestion_count'].sum()).reset_index().rename(columns={0: 'starts_with_suggestion_count'})

report_df = report_df.merge(starts_with_suggestion_count_per_model, on='modelName', how='left')

report_df

/var/folders/54/q9n7_qgd4_98z8qr8wmt28fh0000gs/T/ipykernel_4399/476677242.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  semantic_matches_rate_per_model = llm_results_df.groupby('modelName').apply(lambda x: x['evaluationResult_semantic_match_count'].sum() * 100 / x['evaluationResult_count'].sum() if x['evaluationResult_count'].sum() > 0 else 0)
/var/folders/54/q9n7_qgd4_98z8qr8wmt28fh0000gs/T/ipykernel_4399/476677242.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the 

,modelName,# loc,total_suggestions_generated,avg_suggestions_per_loc,avg_time_in_ms,syntactic_matches,semantic_matches,semantic_match_rate,starts_with_suggestion_count
0,array,8,64,8.000000,19142.618969,4,1,1.562500,5
1,classroom-fol,38,485,12.763158,26219.957100,19,204,42.061856,29
2,courses-v2,40,454,11.350000,21570.513820,18,145,31.938326,40
3,git-redacted,20,117,5.850000,33411.818150,10,12,10.256410,16


In [18]:
report_df.rename(columns={
  "total_suggestions_generated": "Total suggestions generated", "avg_suggestions_per_loc": "Avg suggestions (per loc)",
  "avg_time_in_ms": "Avg time (ms)", 
  "syntactic_matches": "Syn", 
  "semantic_matches": "Sem",
  "semantic_match_rate": "%Valid"
  }, inplace=True)
report_df[report_df['modelName'].isin(['git-redacted', 'array', 'classroom-fol', 'courses-v2'])].style.hide(axis="index")

modelName,# loc,Total suggestions generated,Avg suggestions (per loc),Avg time (ms),Syn,Sem,%Valid,starts_with_suggestion_count
array,8,64,8.000000,19142.618969,4,1,1.562500,5
classroom-fol,38,485,12.763158,26219.957100,19,204,42.061856,29
courses-v2,40,454,11.350000,21570.513820,18,145,31.938326,40
git-redacted,20,117,5.850000,33411.818150,10,12,10.256410,16


In [ ]:
# report_df.to_csv('llm_gpt-4.1_report.csv', index=False)

Bad pipe message: %s [b'\x86\xe8\x03\xc8\xad4\x92\xc5WY\xf4\n\xf2e\xa5\xa6\x92P\x00\x01|\x00\x00\x00\x01\x00\x02\x00\x03\x00\x04\x00\x05\x00\x06\x00\x07\x00\x08\x00\t\x00\n\x00\x0b\x00\x0c\x00\r\x00\x0e\x00\x0f\x00', b'\x11\x00\x12\x00\x13\x00\x14\x00\x15\x00\x16\x00\x17\x00\x18']
Bad pipe message: %s [b'\xdd\x9f\xcb\xcf\x0f\xba#\xa3@\xce;@\xab\x13\x1f\xf8']
Bad pipe message: %s [b'\x00\x01|\x00\x00\x00\x01\x00\x02\x00\x03\x00\x04\x00\x05\x00\x06\x00\x07\x00\x08\x00\t\x00\n\x00\x0b\x00\x0c\x00\r\x00\x0e\x00\x0f\x00\x10\x00\x11\x00\x12\x00\x13\x00\x14\x00\x15\x00\x16\x00\x17\x00\x18\x00\x19\x00\x1a\x00\x1b\x00/\x000\x001\x002\x003\x004\x005\x006\x007\x008\x009\x00:\x00;\x00<\x00=\x00>\x00?\x00@\x00A\x00B\x00C\x00D\x00E\x00F\x00g\x00h\x00i\x00j\x00k\x00l\x00m\x00\x84\x00\x85\x00\x86\x00\x87\x00\x88\x00\x89\x00\x96\x00\x97\x00\x98\x00\x99\x00\x9a\x00\x9b\x00\x9c\x00\x9d\x00\x9e\x00\x9f\x00\xa0\x00\xa1\x00\xa2\x00\xa3\x00\xa4\x00\xa5\x00\xa6\x00\xa7\x00\xba\x00\xbb\x00\xbc\x00\xbd\x00\xbe\